# Phase 2 - Step 1.1: Dataset Cleaning & Enhanced Preprocessing

You were absolutely right—it makes the most sense for **you** to run this locally or on Colab because you have the raw images downloaded and the disk space ready to go!

This notebook covers **Tasks 1, 2, and 4** from our implementation plan:
1. **Data Cleaning:** Removes the 4-5 corrupted images you identified.
2. **Enhanced Preprocessing:** Applies the new **Green Channel CLAHE** to heavily boost the visibility of tiny MAs.
3. **Data Splitting:** Generates the new split CSVs (`idrid`, `maples`, and `combined`).
4. **Export:** Zips the output dataset ready for your training runs.

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import shutil
from pathlib import Path
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import zipfile

zip_path = "DR_dataset_segmentation_processed.zip"     # Change this to your actual zip file name
extract_dir = "dataset"           # Where to extract your files

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
     zip_ref.extractall(extract_dir)

# --- CONFIGURE YOUR PATHS HERE ---
# Where your current extracted dataset lives
BASE_INPUT_DIR = 'dataset'

# Where we will save the new, cleaned, and enhanced dataset
BASE_OUTPUT_DIR = 'dataset_stage1_v2_enhanced'
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(BASE_OUTPUT_DIR, 'images'), exist_ok=True)

### 1. Define Corrupted Images (The Blacklist) 🚫

In [ ]:
# 👉 AYMEN: PASTE THE CORRUPTED IMAGE ROW_NAMES / FILENAMES HERE!
blacklisted_images = [
    'RETINOMIX_RITE_train_image_52','RETINOMIX_RITE_train_image_51','RETINOMIX_RITE_train_image_50','RETINOMIX_RITE_train_image_49','IDRID_test_IDRiD_55','IDRID_test_IDRiD_56','IDRID_test_IDRiD_57','IDRID_test_IDRiD_58','IDRID_test_IDRiD_59','IDRID_test_IDRiD_60','IDRID_test_IDRiD_61','IDRID_test_IDRiD_61','IDRID_test_IDRiD_63','IDRID_test_IDRiD_64','IDRID_test_IDRiD_65','IDRID_test_IDRiD_66',
    'IDRID_test_IDRiD_67','IDRID_test_IDRiD_68','IDRID_test_IDRiD_69','IDRID_test_IDRiD_70','IDRID_test_IDRiD_71','IDRID_test_IDRiD_72','IDRID_test_IDRiD_73','IDRID_test_IDRiD_74','IDRID_test_IDRiD_75','IDRID_test_IDRiD_76','IDRID_test_IDRiD_77','IDRID_test_IDRiD_78','IDRID_test_IDRiD_79','IDRID_test_IDRiD_80','IDRID_test_IDRiD_81',
    'IDRID_train_IDRiD_01','IDRID_train_IDRiD_02','IDRID_train_IDRiD_03','IDRID_train_IDRiD_04','IDRID_train_IDRiD_05','IDRID_train_IDRiD_06','IDRID_train_IDRiD_07','IDRID_train_IDRiD_08','IDRID_train_IDRiD_09','IDRID_train_IDRiD_10','IDRID_train_IDRiD_11','IDRID_train_IDRiD_12','IDRID_train_IDRiD_13','IDRID_train_IDRiD_14','IDRID_train_IDRiD_15','IDRID_train_IDRiD_16','IDRID_train_IDRiD_17','IDRID_train_IDRiD_18','IDRID_train_IDRiD_19','IDRID_train_IDRiD_20','IDRID_train_IDRiD_21','IDRID_train_IDRiD_22','IDRID_train_IDRiD_23','IDRID_train_IDRiD_24','IDRID_train_IDRiD_25','IDRID_train_IDRiD_26','IDRID_train_IDRiD_27','IDRID_train_IDRiD_28','IDRID_train_IDRiD_29','IDRID_train_IDRiD_30','IDRID_train_IDRiD_31','IDRID_train_IDRiD_32','IDRID_train_IDRiD_33','IDRID_train_IDRiD_34','IDRID_train_IDRiD_35','IDRID_train_IDRiD_36','IDRID_train_IDRiD_37','IDRID_train_IDRiD_38','IDRID_train_IDRiD_39','IDRID_train_IDRiD_40',
    'IDRID_train_IDRiD_41','IDRID_train_IDRiD_41','IDRID_train_IDRiD_42','IDRID_train_IDRiD_43','IDRID_train_IDRiD_44','IDRID_train_IDRiD_45','IDRID_train_IDRiD_46','IDRID_train_IDRiD_47','IDRID_train_IDRiD_48','IDRID_train_IDRiD_49','IDRID_train_IDRiD_50','IDRID_train_IDRiD_51','IDRID_train_IDRiD_52','IDRID_train_IDRiD_53','IDRID_train_IDRiD_54'
]

print(f"Loaded {len(blacklisted_images)} corrupted images to exclude.")

### 2. The Enhanced Preprocessing Logic 🔬

In [ ]:
def apply_standard_clahe(image_bgr, clip_limit=2.0, tile_grid=(8, 8)):
    lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB)
    l_ch, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    l_enhanced = clahe.apply(l_ch)
    enhanced_lab = cv2.merge([l_enhanced, a, b])
    return cv2.cvtColor(enhanced_lab, cv2.COLOR_LAB2BGR)

def apply_green_clahe(image_bgr, clip_limit=4.0, tile_grid=(4, 4), invert=True):
    # Extract green channel (index 1 in OpenCV BGR)
    green = image_bgr[:, :, 1]
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    green_enhanced = clahe.apply(green)
    if invert:
        green_enhanced = 255 - green_enhanced
    return green_enhanced

def preprocess_fundus_v2(image_path):
    """
    Reads the original image, applies standard CLAHE, and replaces the
    green channel with our ultra-high-contrast MA-focused CLAHE.
    """
    image = cv2.imread(image_path)
    if image is None:
        return None

    # 1. Base standard enhancement
    base_enhanced = apply_standard_clahe(image)

    # 2. Aggressive green-channel enhancement for MAs
    ma_green = apply_green_clahe(image, clip_limit=4.0, tile_grid=(4, 4), invert=False)

    # 3. Merge: We replace the base green channel with our MA-enhanced one
    # Thus models get standard context + highly highlighted MAs
    base_enhanced[:, :, 1] = ma_green

    return base_enhanced

In [ ]:
def visualize_comparison(image_path):
    img = cv2.imread(image_path)
    if img is None:
        print("Image not found!")
        return
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # V1: Standard CLAHE only
    v1_enhanced = apply_standard_clahe(img)
    v1_rgb = cv2.cvtColor(v1_enhanced, cv2.COLOR_BGR2RGB)

    # V2: Our new method (Standard CLAHE + Green Injection)
    v2_enhanced = preprocess_fundus_v2(image_path)
    v2_rgb = cv2.cvtColor(v2_enhanced, cv2.COLOR_BGR2RGB)

    # Difference (to see exactly what changed)
    diff = cv2.absdiff(v1_enhanced, v2_enhanced)

    plt.figure(figsize=(20, 10))

    plt.subplot(1, 4, 1)
    plt.title("Original")
    plt.imshow(img_rgb)
    plt.axis("off")

    plt.subplot(1, 4, 2)
    plt.title("V1 (Standard CLAHE)")
    plt.imshow(v1_rgb)
    plt.axis("off")

    plt.subplot(1, 4, 3)
    plt.title("V2 (Green Injection)")
    plt.imshow(v2_rgb)
    plt.axis("off")

    plt.subplot(1, 4, 4)
    plt.title("Difference Map (Added Detail)")
    plt.imshow(cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY), cmap="hot")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

# Test on one image
# Replace with a path from your BASE_INPUT_DIR
example_path = os.path.join(BASE_INPUT_DIR, "images/MAPLES_train_20051201_37462_0400_PP.png")
visualize_comparison(example_path)
example_path = os.path.join(BASE_INPUT_DIR, "images/VESSEL_train_Image_08L_CHASE.png")
visualize_comparison(example_path)
example_path = os.path.join(BASE_INPUT_DIR, "images/VESSEL_train_12_dr_HRF.png")
visualize_comparison(example_path)
example_path = os.path.join(BASE_INPUT_DIR, "images/RETINOMIX_RITE_test_image_09.png")
visualize_comparison(example_path)
example_path = os.path.join(BASE_INPUT_DIR, "images/RETINOMIX_DRIVE_train_image_14.png")
visualize_comparison(example_path)

### 3. Build & Clean the Dataset ⚒️
We will loop through the old CSVs, exclude bad images, copy the masks directly to the new folder, and process the images with the new enhanced function.

In [ ]:
splits = ['train_split', 'val_split', 'test_split']
combined_dfs = {s: [] for s in splits}
maples_dfs = {s: [] for s in splits}
drive_dfs = {s: [] for s in splits}
chase_dfs = {s: [] for s in splits}
rite_dfs = {s: [] for s in splits}
hrf_dfs = {s: [] for s in splits}

# Copy masks folder completely, because mask values don't need re-processing
print("Copying masks...")
old_masks_dir = os.path.join(BASE_INPUT_DIR, 'lesion_masks')
new_masks_dir = os.path.join(BASE_OUTPUT_DIR, 'lesion_masks')
if not os.path.exists(new_masks_dir):
    shutil.copytree(old_masks_dir, new_masks_dir)

old_vessel_dir = os.path.join(BASE_INPUT_DIR, 'vessel_masks')
new_vessel_dir = os.path.join(BASE_OUTPUT_DIR, 'vessel_masks')
if not os.path.exists(new_vessel_dir) and os.path.exists(old_vessel_dir):
    shutil.copytree(old_vessel_dir, new_vessel_dir)

# Process Images line by line
for split in splits:
    old_csv_path = os.path.join(BASE_INPUT_DIR, f'{split}.csv')
    df = pd.read_csv(old_csv_path)

    processed_rows = []
    print(f"Processing {split} split...")
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        img_filename = row['img_id']

        # --- 1. FILTER ---
        if img_filename in blacklisted_images or 'IDRiD' in img_filename:
            print(f"Excluded: {img_filename}")
            continue

        # --- 2. ENHANCE ---
        old_img_path = os.path.join(BASE_INPUT_DIR, row['img_path'])
        new_img_rel_path = os.path.join('images', img_filename)
        new_img_full_path = os.path.join(BASE_OUTPUT_DIR, new_img_rel_path)

        if not os.path.exists(new_img_full_path):
            enhanced_img = preprocess_fundus_v2(old_img_path)
            if enhanced_img is not None:
                cv2.imwrite(new_img_full_path, enhanced_img)

        # Update paths if necessary, but assuming relative paths stay identical
        # (e.g., 'images/file.png')
        processed_rows.append(row)

    new_df = pd.DataFrame(processed_rows)
    combined_dfs[split] = new_df

    # Split by explicit dataset categories
    maples_dfs[split] = new_df[new_df['img_id'].str.contains('MAPLES|MESSIDOR', case=False, na=False)]
    drive_dfs[split] = new_df[new_df['img_id'].str.contains('DRIVE', case=False, na=False)]
    chase_dfs[split] = new_df[new_df['img_id'].str.contains('CHASE', case=False, na=False)]
    rite_dfs[split] = new_df[new_df['img_id'].str.contains('RITE', case=False, na=False)]
    hrf_dfs[split] = new_df[new_df['img_id'].str.contains('HRF', case=False, na=False)]

    # Save the combined CSV
    new_df.to_csv(os.path.join(BASE_OUTPUT_DIR, f'combined_{split}.csv'), index=False)

### 4. Create Source-Specific CSVs 📁
For Task 5 (training all models on individual datasets), we need these standalone split files.

In [ ]:
for split in splits:
    if len(maples_dfs[split]) > 0:
        maples_dfs[split].to_csv(os.path.join(BASE_OUTPUT_DIR, f'maples_{split}.csv'), index=False)
    if len(drive_dfs[split]) > 0:
        drive_dfs[split].to_csv(os.path.join(BASE_OUTPUT_DIR, f'drive_{split}.csv'), index=False)
    if len(chase_dfs[split]) > 0:
        chase_dfs[split].to_csv(os.path.join(BASE_OUTPUT_DIR, f'chase_{split}.csv'), index=False)
    if len(rite_dfs[split]) > 0:
        rite_dfs[split].to_csv(os.path.join(BASE_OUTPUT_DIR, f'rite_{split}.csv'), index=False)
    if len(hrf_dfs[split]) > 0:
        hrf_dfs[split].to_csv(os.path.join(BASE_OUTPUT_DIR, f'hrf_{split}.csv'), index=False)

print("All CSVs created successfully!")

### 5. Zip & Download 📦

In [ ]:
!zip -r dataset_stage1_v2_enhanced.zip dataset_stage1_v2_enhanced/
print("Done! You can now download dataset_stage1_v2_enhanced.zip and upload it to Kaggle for the Phase 2 training.")